# RAG FDP Notebook - Step by Step

In [ ]:
!pip -q install sentence-transformers faiss-cpu pymupdf transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 50.3 MB/s eta 0:00:00


## Cell 1: Upload PDF

In [ ]:
from google.colab import files
uploaded=files.upload()
pdf_file=list(uploaded.keys())[0]

Saving HR.pdf to HR.pdf


## Cell 2: Extract Text

In [ ]:
import fitz
text=''
doc=fitz.open(pdf_file)
for p in doc:
    text+=p.get_text()
print("Characters:", len(text))
print("Words:", len(text.split()))
print("Pages:", len(doc))

Characters: 45755
Words: 7225
Pages: 17


## Cell 3: Chunking

In [ ]:
chunks=[text[i:i+500] for i in range(0,len(text),500)]
print(len(chunks))

92


## Cell 4: Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings=model.encode(chunks)
print(embeddings.shape)

## Cell 5: FAISS

In [ ]:
import faiss,numpy as np
index=faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

## Cell 6: Retrieval

In [ ]:
query='What is the attendance requirement?'
q=model.encode([query])
D,I=index.search(np.array(q).astype('float32'),3)
for idx in I[0]:
 print(chunks[idx][:200])

## Cell 7: FLAN-T5

In [ ]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
tokenizer=AutoTokenizer.from_pretrained('google/flan-t5-large')
llm=AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large')

## Cell 8: Final RAG Answer

In [ ]:
context='\n'.join([chunks[i] for i in I[0]])
prompt=f'Context:\n{context}\n\nQuestion:{query}\nAnswer:'
inputs=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=1024)
out=llm.generate(**inputs,max_new_tokens=150)
print(tokenizer.decode(out[0],skip_special_tokens=True))